# Промпт #19 — Комбинация техник

**Техника:** System prompt + Few-shot + CoT + JSON вывод  
**Задача:** Анализ отзыва — тональность, причина, рекомендация в JSON  
**Сложность:** ⭐⭐⭐⭐☆


In [1]:
import sys
sys.path.append('..')
from config import get_completion
import json

In [2]:
system = """Ты эксперт по анализу отзывов клиентов. 
Всегда думай пошагово перед ответом.
Отвечай только валидным JSON без markdown-фенсов."""

prompt = """Проанализируй отзыв и верни JSON с полями:
- тональность: ПОЗИТИВНЫЙ / НЕГАТИВНЫЙ / НЕЙТРАЛЬНЫЙ
- причина: одно предложение почему
- рекомендация: что сделать бизнесу

Примеры:
<examples>
<example>
Отзыв: "Доставка быстрая, товар exactly как на фото!"
Ответ: {"тональность": "ПОЗИТИВНЫЙ", "причина": "Клиент доволен скоростью и качеством", "рекомендация": "Поддерживать текущий уровень сервиса"}
</example>
<example>
Отзыв: "Ждал месяц, привезли не то что заказывал"
Ответ: {"тональность": "НЕГАТИВНЫЙ", "причина": "Долгая доставка и ошибка в заказе", "рекомендация": "Улучшить логистику и контроль сборки заказов"}
</example>
</examples>

Отзыв: "Качество неплохое, но цена завышена раза в два. Взял только потому что срочно нужно было." """

response = get_completion(prompt, system_prompt=system, temperature=0.1)
print("=== ОТВЕТ МОДЕЛИ ===")
print(response)

try:
    clean = response.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    parsed = json.loads(clean)
    print("\n✅ Валидный JSON")
    print(f"Тональность: {parsed['тональность']}")
    print(f"Причина: {parsed['причина']}")
    print(f"Рекомендация: {parsed['рекомендация']}")
except json.JSONDecodeError:
    print("\n❌ Невалидный JSON")

=== ОТВЕТ МОДЕЛИ ===
{"тональность": "НЕЙТРАЛЬНЫЙ", "причина": "Клиент удовлетворен качеством, но считает цену слишком высокой", "рекомендация": "Пересмотреть ценовую политику и рассмотреть возможность скидок или акций"}

✅ Валидный JSON
Тональность: НЕЙТРАЛЬНЫЙ
Причина: Клиент удовлетворен качеством, но считает цену слишком высокой
Рекомендация: Пересмотреть ценовую политику и рассмотреть возможность скидок или акций


## Оценка: 5/5

## Инсайт
Комбинация сработала идеально — валидный JSON с первого раза, без фенсов,
тональность определена верно (отзыв действительно неоднозначный: качество ок, 
цена плохая → НЕЙТРАЛЬНЫЙ).

Что сделала каждая техника:
- System prompt → задал роль эксперта и запретил markdown-фенсы (сработало!)
- Few-shot примеры → показали точный формат JSON который нужен
- temperature=0.1 → стабильный структурированный вывод
- Неоднозначный отзыв → модель не "срезала путь" в НЕГАТИВНЫЙ, 
  взвесила оба аспекта

Главный вывод: когда техники работают вместе, каждая закрывает слабость другой.
System prompt убрал фенсы (проблема из #18), few-shot задал формат (урок из #13),
низкая температура зафиксировала структуру. Сборка > сумма частей.
